In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# Define reference timestamp for relative time parsing
reference_timestamp = to_timestamp(lit("2026-08-14 15:00:00"))

# Read bronze table
bronze_df = spark.table("amit.default.donations_bronze")

# Deduplicate by prisoner and donation_id (keep first occurrence)
dedup_df = bronze_df.dropDuplicates(["prisoner", "donation_id"])

# Transform the data
silver_df = dedup_df.select(
    col("donation_id"),
    
    # Transform donor_name to lowercase
    lower(col("donor_name")).alias("donor_name"),
    
    # Parse time_text to timestamp
    # Hebrew month patterns with dynamic year calculation
    when(
        col("time_text").rlike(r"\d+\s+ב(ינו|פבר|מרץ|אפר|מאי|יוני|יולי|אוג|ספט|אוק|נוב|דצמ)"),
        to_timestamp(
            concat(
                # Year logic: months > 9 (Oct-Dec) = 2025, months <= 9 (Jan-Sep) = 2026
                when(
                    col("time_text").rlike(r"ב(אוק|נוב|דצמ)"),
                    lit("2025")
                ).otherwise(lit("2026")),
                lit("-"),
                # Month mapping
                when(col("time_text").like("%בינו%"), lit("01"))
                .when(col("time_text").like("%בפבר%"), lit("02"))
                .when(col("time_text").like("%במרץ%"), lit("03"))
                .when(col("time_text").like("%באפר%"), lit("04"))
                .when(col("time_text").like("%במאי%"), lit("05"))
                .when(col("time_text").like("%ביוני%"), lit("06"))
                .when(col("time_text").like("%ביולי%"), lit("07"))
                .when(col("time_text").like("%באוג%"), lit("08"))
                .when(col("time_text").like("%בספט%"), lit("09"))
                .when(col("time_text").like("%באוק%"), lit("10"))
                .when(col("time_text").like("%בנוב%"), lit("11"))
                .when(col("time_text").like("%בדצמ%"), lit("12")),
                lit("-"),
                # Day
                lpad(regexp_extract(col("time_text"), r"(\d+)\s+ב", 1), 2, "0"),
                lit(" "),
                # Hour
                lpad(regexp_extract(col("time_text"), r"בשעה\s+(\d+):(\d+)", 1), 2, "0"),
                lit(":"),
                # Minute
                lpad(regexp_extract(col("time_text"), r"בשעה\s+(\d+):(\d+)", 2), 2, "0"),
                lit(":00")
            ),
            "yyyy-MM-dd HH:mm:ss"
        )
    ).when(
        # Pattern: "לפני X שעות" (X hours ago)
        col("time_text").like("%שעות%"),
        reference_timestamp - (regexp_extract(col("time_text"), r"(\d+)", 1).cast("int") * expr("INTERVAL 1 HOUR"))
    ).when(
        # Pattern: "לפני שעתיים" (2 hours ago)
        col("time_text").like("%שעתיים%"),
        reference_timestamp - expr("INTERVAL 2 HOURS")
    ).when(
        # Pattern: "לפני שעה" (1 hour ago)
        col("time_text").like("%שעה%") & ~col("time_text").like("%שעות%") & ~col("time_text").like("%שעתיים%"),
        reference_timestamp - expr("INTERVAL 1 HOUR")
    ).when(
        # Pattern: "לפני X דקות" (X minutes ago)
        col("time_text").like("%דקות%"),
        reference_timestamp - (regexp_extract(col("time_text"), r"(\d+)", 1).cast("int") * expr("INTERVAL 1 MINUTE"))
    ).when(
        # Pattern: "לפני דקה" (1 minute ago)
        col("time_text").like("%דקה%") & ~col("time_text").like("%דקות%"),
        reference_timestamp - expr("INTERVAL 1 MINUTE")
    ).otherwise(None).alias("donation_timestamp"),
    
    # Extract numeric amount (remove ₪ and other non-numeric characters, convert to int)
    regexp_replace(col("amount_text"), r"[^\d]", "").cast("int").alias("amount_ils"),
    
    # Keep original text fields for reference
    col("time_text"),
    col("amount_text"),
    col("comment_text"),
    
    # Metadata
    col("prisoner"),
    col("source_file"),
    col("ingestion_timestamp")
)

# Calculate first donation time per prisoner and hours since first donation
window_prisoner = Window.partitionBy("prisoner").orderBy("donation_timestamp")

silver_with_calcs_df = silver_df.withColumn(
    "first_donation_timestamp",
    first_value("donation_timestamp").over(
        Window.partitionBy("prisoner").orderBy("donation_timestamp").rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)
    )
).withColumn(
    "hours_since_first_donation",
    floor(
        (unix_timestamp("donation_timestamp") - unix_timestamp("first_donation_timestamp")) / 3600
    ).cast("int")
)

# Write to silver table
silver_with_calcs_df.write \
    .mode("overwrite") \
    .saveAsTable("amit.default.donations_silver")

print(f"✓ Silver table created successfully")
print(f"  Total donations: {silver_with_calcs_df.count()}")
print(f"  Prisoners: {silver_with_calcs_df.select('prisoner').distinct().count()}")

In [0]:
# Display sample records from silver table
display(
    spark.table("amit.default.donations_silver")
        .orderBy("prisoner", "donation_timestamp")
        .limit(50)
)

In [0]:
# Check for any unparsed timestamps
unparsed_timestamps = spark.table("amit.default.donations_silver") \
    .filter("donation_timestamp IS NULL") \
    .select("donation_id", "prisoner", "time_text")

print(f"Records with unparsed timestamps: {unparsed_timestamps.count()}")
if unparsed_timestamps.count() > 0:
    print("\nSample unparsed time_text values:")
    display(unparsed_timestamps.limit(10))

# Check for any unparsed amounts
unparsed_amounts = spark.table("amit.default.donations_silver") \
    .filter("amount_ils IS NULL OR amount_ils = 0") \
    .select("donation_id", "prisoner", "amount_text", "amount_ils")

print(f"\nRecords with unparsed amounts: {unparsed_amounts.count()}")
if unparsed_amounts.count() > 0:
    print("\nSample unparsed amount_text values:")
    display(unparsed_amounts.limit(10))

In [0]:
# Summary statistics per prisoner
summary_df = spark.table("amit.default.donations_silver") \
    .groupBy("prisoner") \
    .agg(
        count("*").alias("total_donations"),
        sum("amount_ils").alias("total_amount_ils"),
        avg("amount_ils").alias("avg_donation_ils"),
        min("amount_ils").alias("min_donation_ils"),
        max("amount_ils").alias("max_donation_ils"),
        expr("percentile_approx(amount_ils, 0.25)").alias("p25_donation_ils"),
        expr("percentile_approx(amount_ils, 0.5)").alias("median_donation_ils"),
        expr("percentile_approx(amount_ils, 0.75)").alias("p75_donation_ils"),
        expr("percentile_approx(amount_ils, 0.9)").alias("p90_donation_ils"),
        expr("percentile_approx(amount_ils, 0.99)").alias("p99_donation_ils"),
        min("donation_timestamp").alias("first_donation"),
        max("donation_timestamp").alias("last_donation"),
        max("hours_since_first_donation").alias("campaign_duration_hours"),
        (max("hours_since_first_donation") / 24).cast("int").alias("campaign_duration_days")
    ) \
    .orderBy("prisoner")

display(summary_df)

In [0]:
# Analyze cumulative donation totals by hours since first donation
from pyspark.sql import Window

hourly_pattern = spark.table("amit.default.donations_silver") \
    .groupBy("prisoner", "hours_since_first_donation") \
    .agg(
        count("*").alias("donation_count"),
        sum("amount_ils").alias("total_amount")
    )

window_spec = Window.partitionBy("prisoner").orderBy("hours_since_first_donation").rowsBetween(Window.unboundedPreceding, Window.currentRow)

hourly_cumulative = hourly_pattern.withColumn(
    "cumulative_total_amount",
    sum("total_amount").over(window_spec)
).orderBy("prisoner", "hours_since_first_donation")

display(hourly_cumulative)